In this notebook we use the network learned on images cropped in the bounding boxes to generate a new dataset. From the original data set `Imagenet_full` we will fixate on the most likely position and generate the `Imagenet_focus` dataset




In [ ]:
import retinoto_py as fovea
args = fovea.Params(do_fovea=True, batch_size=1, shuffle=False)
print(args)

In [ ]:
resolution = (100, 100)
resolution = (30, 30)
resolution = (7, 7)
resolution = (25, 25)
resolution = (21, 21)
size_ratio = .75
size_ratio = .45
size_ratio = .618
sigma = .15
sigma = .0

## Building the new dataset


In [ ]:
FOCUS_DATA_DIR = args.DATAROOT / 'Imagenet_focus'
FOCUS_DATA_DIR.mkdir(exist_ok=True)

IMG_EXTS = {'.jpg', '.jpeg', '.JPEG', '.png', '.bmp'}
def clean_list(list_dir, EXCLUDED_FILES={'.DS_Store', '.ipynb_checkpoints'}):
    return [ p for p in list_dir if p.is_file() and p.name not in EXCLUDED_FILES  ]
# parameters for the new dataset
format = 'png'

from torchvision.transforms.functional import InterpolationMode, resize
subset_factor = 50

args = fovea.Params(do_fovea=True, model_name='convnext_base', batch_size=1, shuffle=False, subset_factor=subset_factor)
model = fovea.load_model(args, model_filename=args.data_cache / f'32_fovea_model_name=convnext_base_dataset=bbox.pth')

for folder in ['val', 'train']:
    print(f'\n Scanning folder "{folder}"')
    DATA_DIR = args.DATAROOT / 'Imagenet_full' / folder
    loader = fovea.get_loader(args, fovea.get_dataset(args, DATA_DIR, do_full_preprocess=False))

    src_root = FULL_DATA_DIR / folder
    tgt_root = FOCUS_DATA_DIR / folder
    tgt_root.mkdir(parents=True, exist_ok=True)

    count_in = 0
    count_out = 0

    # parcours récursif avec pathlib
    for img_path in fovea.tqdm(clean_list(list(src_root.rglob('*.*')))):
        if not img_path.is_file() or img_path.suffix not in IMG_EXTS:
            print(f'File {img_path} is detected as an invalid image.')
            continue

        count_in += 1
        imgid = img_path.stem
        # Get the list of bounding boxes for a specific image. If that list is empty (meaning no boxes were found for this image), then skip to the next image and don't run any code that comes after this line.
        boxes = get_boxes(df_data, imgid)
        if not boxes:
            continue

        class_id = img_path.parent.name
        true_idx = class_to_idx[class_id]
        target_folder = tgt_root / class_id
        target_folder.mkdir(parents=True, exist_ok=True)

        original_image = None
        for i_obj, b in enumerate(boxes):
            no = '' if i_obj == 0 else f'_{i_obj}'
            out_path = target_folder / f'{imgid}{no}.{format}'
            if out_path.is_file():
                # the file already exists let's skip it
                count_out += 1
                continue

            if original_image is None:
                try:
                    # original_image = Image.open(img_path).convert('RGB')
                    image = read_image(img_path)/255.
                    if image.shape[0] == 1: image = image.repeat(3, 1, 1)
                    three, H, W = image.shape
                    assert three == 3

                except Exception as e:
                    print(f' could not open {img_path}: {e}')
                    break





        n_dataset = len(dataset)
        likelihood_maps_label = fovea.np.empty((resolution[0], resolution[1], 0))
        likelihood_maps_max = fovea.np.empty((resolution[0], resolution[1], 0))
        for _, (image, true_idx) in fovea.tqdm(enumerate(loader), total=n_dataset):
            image, true_idx = image.to(args.device), true_idx.to(args.device)
            image = image.squeeze(0)
            three, H, W = image.shape

            if max((H, W)) > image_size_full:
                image = resize(image, image_size_full, interpolation=InterpolationMode.BILINEAR, antialias=True)

            pos_H, pos_W = fovea.get_positions(H, W, resolution=resolution)

            try:
                probas = fovea.compute_likelihood_map(args, model, image, pos_H, pos_W, size_ratio=size_ratio)
                likelihood_maps_label_ = probas[:, true_idx].cpu().numpy().reshape(resolution)
                likelihood_maps_label = fovea.np.dstack((likelihood_maps_label, likelihood_maps_label_))
                proba_max, _ = probas.max(axis=-1)
                likelihood_maps_max_ = proba_max.cpu().numpy().reshape(resolution)
                likelihood_maps_max = fovea.np.dstack((likelihood_maps_max, likelihood_maps_max_))
            except Exception as e:
                print(e)#, probas.shape, image.shape)


            proba_max, idx = fovea.torch.max(likelihood_maps_label[:, :, count])


            count += 1



            box_size = int( fovea.np.sqrt(H*W)*size_ratio)
            xmin, ymin, xmax, ymax = b['xmin'], b['ymin'], b['xmax'], b['ymax']
            W_box = xmax - xmin
            H_box = ymax - ymin

            pos_H, pos_W = fovea.get_positions(H_box, W_box, resolution=resolution)
            pos_H, pos_W = ymin + pos_H, xmin + pos_W
            
            probas = fovea.compute_likelihood_map(args, model, original_image, pos_H, pos_W, size_ratio=size_ratio)
        
            likelihood_map = probas[:, true_idx]
            if sigma > 0: likelihood_map = gaussian_filter(likelihood_map, sigma=sigma)

            likelihood_max, idx_pos = likelihood_map.max(axis=-1)
            
            image_fix = fovea.fixate(original_image, pos_H[idx_pos], pos_W[idx_pos], box_size) 

            img_pil = fovea.TF.to_pil_image(image_fix)
            img_pil.save(out_path, format=format)

            count_out += 1

    print(f' - in: {count_in} / out: {count_out}')

Voilà !